# Lab 1: LLM Inference, Decoding, and Structured Output
**EECE.4860/5860 Introduction to Generative AI, Spring 2027**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/labs/lab1/lab1.ipynb)

In this lab you will:
* **Part A (all students):** compare tokenizers, implement temperature and top-p (nucleus) filtering, and measure how decoding settings change the diversity of generated text.
* **Part B (all students):** get a small LLM to return valid JSON for an information-extraction task, comparing zero-shot, few-shot, and validate-and-retry prompting.
* **Part C (EECE.5860 only):** measure how much *test-time compute* (reasoning tokens, self-consistency) improves math accuracy.

**Before you start**
1. *File > Save a copy in Drive*, so your work is saved.
2. *Runtime > Change runtime type > T4 GPU*.
3. Fill in your student ID digits and section in the first code cell, then run the cells top to bottom.

**Time budget on a Colab T4:** about 15 min of GPU time for Parts A and B, plus about 10 min for Part C. Models used: [Qwen3-0.6B and Qwen3-1.7B](https://huggingface.co/Qwen) (Apache-2.0, no access request needed).

**Grading (15 points):** 2 for a notebook that runs top to bottom with all outputs saved, 8 for the `[CHECK ...]` lines, and 5 for the report section at the end. EECE.5860: Part C adds 5 points and the total is scaled to 15.

**Submit on Canvas:** (1) this notebook with outputs (*File > Download > Download .ipynb*) and (2) a PDF of it (*File > Print > Save as PDF*). Do not delete or edit the `[CHECK ...]` or `GENAI-RESULTS` output lines; the grader reads them.

## 0. Setup

In [ ]:
STUDENT_ID_LAST4 = ""   # last 4 digits of your student ID, e.g. "0427"
SECTION = "4860"        # "4860" or "5860"

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:   # course helper package
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')

import json, math, os, random, re, collections
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from genai_utils import setup_env, Checker, variant, measure

QUICK = os.environ.get("GENAI_QUICK") == "1"   # instructor smoke test only; leave as is
device, dtype = setup_env(seed=int(STUDENT_ID_LAST4 or 0))
ck = Checker("lab1", STUDENT_ID_LAST4, SECTION)
V = variant(STUDENT_ID_LAST4, 4)               # your personal variant (0-3)
SMALL_MODEL = "Qwen/Qwen3-0.6B"
MAIN_MODEL = SMALL_MODEL if QUICK else "Qwen/Qwen3-1.7B"
DATA_URL = "https://raw.githubusercontent.com/acanets/genai-labs/spring27/labs/lab1/data/events.jsonl"
print("Your variant:", V)

In [ ]:
# Helper functions used throughout the lab (read them, no changes needed).
_models = {}

def load(model_id):
    """Load (and cache) a tokenizer and causal LM in inference mode."""
    if model_id not in _models:
        tok = AutoTokenizer.from_pretrained(model_id)
        tok.padding_side = "left"          # needed for batched generation with decoder-only models
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(device).eval()
        _models[model_id] = (tok, model)
    return _models[model_id]

def chat_prompts(tok, user_msgs, thinking=False, system=None, history=None):
    """Turn user messages into model-ready prompt strings using the model's chat template."""
    texts = []
    for i, m in enumerate(user_msgs):
        msgs = [{"role": "system", "content": system}] if system else []
        if history is not None:
            msgs += history[i]
        msgs.append({"role": "user", "content": m})
        texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                             enable_thinking=thinking))
    return texts

@torch.no_grad()
def generate(model_id, prompts, max_new_tokens=64, batch_size=32, **gen):
    """Batched generation. Returns (texts, number_of_generated_tokens)."""
    tok, model = load(model_id)
    texts, counts = [], []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **gen)
        new = out[:, enc["input_ids"].shape[1]:]
        texts += tok.batch_decode(new, skip_special_tokens=True)
        counts += [int((row != tok.pad_token_id).sum()) for row in new]
    return texts, counts

## Part A: Tokens and decoding (all students)

### A1. Tokenizers
Different models split the same text into different numbers of tokens. Token counts determine cost, speed, and how much fits in the context window.

**Task:** add one text of your own to `TEXTS` (any language or domain: a sentence in your native language, a chemical formula, an emoji-heavy message, ...).

In [ ]:
TOKENIZERS = {
    "GPT-2 (byte-level BPE)": "openai-community/gpt2",
    "BERT (WordPiece)": "google-bert/bert-base-uncased",
    "Qwen3 (byte-level BPE)": SMALL_MODEL,
}
TEXTS = {
    "english": "Generative models learn a probability distribution over data and sample from it.",
    "code": "def add(a, b):\n    return a + b  # simple function",
    "spanish": "Los modelos generativos aprenden una distribución de probabilidad.",
    "numbers": "In 2027, 3.14159 x 2,000,000 = 6,283,180.",
}
TEXTS["mine"] = ""   # TODO: your own text

toks = {name: AutoTokenizer.from_pretrained(mid) for name, mid in TOKENIZERS.items()}
table = {name: {k: len(t.encode(v, add_special_tokens=False)) for k, v in TEXTS.items()} for name, t in toks.items()}
print(f"{'tokenizer':26s}" + "".join(f"{k:>10s}" for k in TEXTS))
for name, row in table.items():
    print(f"{name:26s}" + "".join(f"{row[k]:>10d}" for k in TEXTS))
print("\nQwen3 tokens for your text:", toks["Qwen3 (byte-level BPE)"].tokenize(TEXTS["mine"]))
ck.expect_true("A.tokenizers", len(TEXTS) == 5 and TEXTS.get("mine", "").strip() != ""
               and all(c > 0 for row in table.values() for c in row.values()), value=table["Qwen3 (byte-level BPE)"])
for name, row in table.items():
    ck.record(f"A.tokens.{name.split()[0]}", row)

### A2. Temperature and top-p, by hand
A language model outputs logits $z_i$ for every token in the vocabulary. Sampling with temperature $T$ uses
$p_i = \dfrac{e^{z_i/T}}{\sum_j e^{z_j/T}}$. Top-p (nucleus) sampling then keeps the **smallest** set of most-probable tokens whose cumulative probability is **at least** $p$, renormalizes, and samples from that set (Holtzman et al., 2020).

**Task:** implement both functions in plain Python (no PyTorch needed).

In [ ]:
def softmax_with_temperature(logits, T):
    """Return a list of probabilities softmax(logits / T)."""
    # TODO: your code here
    raise NotImplementedError

def top_p_indices(probs, p):
    """Return the sorted list of indices kept by top-p filtering with threshold p."""
    # TODO: your code here
    raise NotImplementedError

ck.expect_fn("A.softmax_T", softmax_with_temperature, [
    (([2.0, 1.0, 0.0], 1.0), [0.6652, 0.2447, 0.0900]),
    (([2.0, 1.0, 0.0], 0.5), [0.8668, 0.1173, 0.0159]),
    (([2.0, 1.0, 0.0], 2.0), [0.5065, 0.3072, 0.1863]),
], tol=1e-3)
ck.expect_fn("A.top_p", top_p_indices, [
    (([0.5, 0.3, 0.15, 0.05], 0.75), [0, 1]),
    (([0.5, 0.3, 0.15, 0.05], 0.5), [0]),
    (([0.5, 0.3, 0.15, 0.05], 0.9), [0, 1, 2]),
    (([0.1, 0.6, 0.3], 0.65), [1, 2]),
])

### A3. Decoding and diversity
We sample 10 completions of the same prompt at several temperatures and measure **distinct-2**: the number of *unique* word bigrams divided by the *total* number of bigrams across all samples (higher = more diverse).

**Task:** implement `distinct_n`, then run the experiment.

In [ ]:
def distinct_n(texts, n):
    """Unique n-grams / total n-grams over all texts (lowercased, whitespace tokens). 0.0 if no n-grams."""
    # TODO: your code here
    raise NotImplementedError

ck.expect_fn("A.distinct_n", distinct_n, [
    ((["a b c", "a b d"], 2), 0.75),
    ((["a b c", "a b d"], 1), 4 / 6),
    ((["one"], 2), 0.0),
], tol=1e-6)

In [ ]:
PROMPTS = [
    "Write one sentence describing a rainy afternoon in a city.",
    "Write one sentence describing the first day of a new job.",
    "Write one sentence describing a robot learning to cook.",
    "Write one sentence describing a train arriving at night.",
]
prompt = chat_prompts(load(SMALL_MODEL)[0], [PROMPTS[V]])
N = 4 if QUICK else 10
results = {}
with measure("A3 sampling", results):
    for T in (0.2, 0.7, 1.2):
        outs, _ = generate(SMALL_MODEL, prompt, max_new_tokens=40, do_sample=True, temperature=T,
                           top_p=1.0, top_k=0, num_return_sequences=N)
        results[f"T={T}"] = outs
    for p in (0.5, 0.95):
        outs, _ = generate(SMALL_MODEL, prompt, max_new_tokens=40, do_sample=True, temperature=1.0,
                           top_p=p, top_k=0, num_return_sequences=N)
        results[f"T=1.0,top_p={p}"] = outs
    greedy1, _ = generate(SMALL_MODEL, prompt, max_new_tokens=40, do_sample=False)
    greedy2, _ = generate(SMALL_MODEL, prompt, max_new_tokens=40, do_sample=False)

for setting in [k for k in results if k.startswith("T=")]:
    d2 = distinct_n(results[setting], 2)
    ck.record(f"A.distinct2[{setting}]", round(d2, 3))
    print(f"{setting:18s} distinct-2 = {d2:.3f} | e.g. {results[setting][0][:90]!r}")
print("greedy:", greedy1[0][:120])
ck.expect_true("A.greedy_deterministic", greedy1 == greedy2)
ck.expect_true("A.temperature_diversity",
               distinct_n(results["T=1.2"], 2) > distinct_n(results["T=0.2"], 2),
               value=(round(distinct_n(results["T=0.2"], 2), 3), round(distinct_n(results["T=1.2"], 2), 3)))

## Part B: Structured output (all students)
Applications need machine-readable output. Your task: extract **date, start time, city, and admission price** from short event announcements as JSON that matches `SCHEMA`. You will compare three strategies on 20 test texts:
1. **zero-shot**: instructions only;
2. **few-shot**: instructions plus worked examples;
3. **few-shot + validate-and-retry**: parse and validate the output; if it is invalid, show the model the error and ask again (once).

We use greedy decoding so results are reproducible.

In [ ]:
import urllib.request
from jsonschema import Draft202012Validator

SCHEMA = {
    "type": "object",
    "properties": {
        "date": {"type": "string", "pattern": r"^\d{4}-\d{2}-\d{2}$"},
        "time": {"type": "string", "pattern": r"^\d{2}:\d{2}$"},
        "city": {"type": "string", "minLength": 1},
        "price_usd": {"type": "number", "minimum": 0},
    },
    "required": ["date", "time", "city", "price_usd"],
    "additionalProperties": False,
}
validator = Draft202012Validator(SCHEMA)

if os.path.exists("data/events.jsonl"):
    rows = [json.loads(l) for l in open("data/events.jsonl")]
else:
    rows = [json.loads(l) for l in urllib.request.urlopen(DATA_URL).read().decode().splitlines()]
EXAMPLES = [r for r in rows if r["split"] == "example"]
TEST = [r for r in rows if r["split"] == "test"][: 5 if QUICK else None]
print(len(EXAMPLES), "examples,", len(TEST), "test texts. First test text:\n", TEST[0]["text"])

def score(preds, rows):
    """Provided evaluation: validity rate and per-field accuracy (invalid outputs count as wrong)."""
    fields = {"date": 0, "time": 0, "city": 0, "price_usd": 0}
    for p, r in zip(preds, rows):
        if p is None:
            continue
        g = r["label"]
        fields["date"] += p["date"] == g["date"]
        fields["time"] += p["time"] == g["time"]
        fields["city"] += p["city"].split(",")[0].strip().casefold() == g["city"].casefold()
        fields["price_usd"] += abs(float(p["price_usd"]) - g["price_usd"]) < 0.01
    n = len(rows)
    out = {"valid": sum(p is not None for p in preds) / n}
    out.update({k: v / n for k, v in fields.items()})
    out["field_acc"] = sum(fields.values()) / (4 * n)
    return out

### B1. Parse and validate
Models often wrap JSON in prose or in a Markdown code fence. Implement `parse_and_validate(text)`, which returns `(obj, None)` for a valid object and `(None, error_message)` otherwise. Hint: find the JSON object in the text, `json.loads` it, then check `validator.iter_errors(obj)`.

In [ ]:
def parse_and_validate(text):
    # TODO: your code here
    raise NotImplementedError

_ok = lambda s: parse_and_validate(s)[0] is not None
ck.expect_fn("B.parse_and_validate", _ok, [
    (('```json\n{"date": "2027-01-02", "time": "09:00", "city": "Lowell", "price_usd": 0}\n```',), True),
    (('Here you go: {"date": "2027-01-02", "time": "09:00", "city": "Lowell", "price_usd": 5.5} Thanks!',), True),
    (('{"date": "Jan 2", "time": "09:00", "city": "Lowell", "price_usd": 0}',), False),
    (('{"date": "2027-01-02", "time": "09:00", "city": "Lowell"}',), False),
    (('not json at all',), False),
    (('{"date": "2027-01-02", "time": "9:00", "city": "Lowell", "price_usd": "free"}',), False),
])

### B2. Zero-shot and few-shot prompts
Write the instruction (`INSTRUCTION`) and a function that builds a few-shot prompt from `EXAMPLES` (3 labeled texts). Tell the model exactly what format each field must have (look at `SCHEMA`), and to output only JSON.

In [ ]:
INSTRUCTION = "Extract the event information from the text as JSON."   # TODO: improve this

def zero_shot(text):
    return f"{INSTRUCTION}\n\nText: {text}"

def few_shot(text):
    # TODO: your code here
    raise NotImplementedError

def run(prompt_fn, rows):
    tok, _ = load(MAIN_MODEL)
    outs, _ = generate(MAIN_MODEL, chat_prompts(tok, [prompt_fn(r["text"]) for r in rows]),
                       max_new_tokens=80, do_sample=False)
    return outs, [parse_and_validate(o)[0] for o in outs]

with measure("B2 zero/few-shot"):
    zs_raw, zs = run(zero_shot, TEST)
    fs_raw, fs = run(few_shot, TEST)
print("zero-shot output for text 0:", zs_raw[0][:200])
print("few-shot  output for text 0:", fs_raw[0][:200])

### B3. Validate and retry
Implement `few_shot_with_retry(rows)`: run the few-shot prompt; for every output that fails `parse_and_validate`, continue the **same conversation**: add the model's answer as an `assistant` message, then a `user` message containing the validation error and a request to return only a corrected JSON object. Retry once. Use `chat_prompts(tok, msgs, history=...)` and `generate(..., do_sample=False)`.

In [ ]:
def few_shot_with_retry(rows, first_raw):
    """first_raw: the few-shot raw outputs you already generated (reuse them; do not regenerate)."""
    tok, _ = load(MAIN_MODEL)
    preds = [parse_and_validate(o)[0] for o in first_raw]
    # TODO: your code here
    raise NotImplementedError
    return preds

with measure("B3 retry"):
    retry = few_shot_with_retry(TEST, fs_raw)

scores = {"zero-shot": score(zs, TEST), "few-shot": score(fs, TEST), "few-shot+retry": score(retry, TEST)}
print(f"{'method':16s}{'valid':>7s}{'date':>7s}{'time':>7s}{'city':>7s}{'price':>7s}{'all':>7s}")
for m, s in scores.items():
    print(f"{m:16s}" + "".join(f"{s[k]:7.2f}" for k in ("valid", "date", "time", "city", "price_usd", "field_acc")))
    ck.record(f"B.{m}", {k: round(v, 3) for k, v in s.items()})
best = max(scores.values(), key=lambda s: s["field_acc"])
ck.expect_true("B.retry_not_worse", scores["few-shot+retry"]["valid"] >= scores["few-shot"]["valid"],
               value=(scores["few-shot"]["valid"], scores["few-shot+retry"]["valid"]))
ck.expect_range("B.best_valid_rate", best["valid"], 0.8, 1.0)
ck.expect_range("B.best_field_accuracy", best["field_acc"], 0.6, 1.0)

### B4. Knowledge cutoff
Run the cell and keep the output; you will discuss it in the report (Q5).

In [ ]:
tok, _ = load(MAIN_MODEL)
ans, _ = generate(MAIN_MODEL, chat_prompts(tok, ["Who won the 2026 World Series? Answer in one sentence."]),
                  max_new_tokens=60, do_sample=False)
print(ans[0])
ck.record("B.world_series_2026_answer", ans[0][:200])

## Part C: Test-time compute (EECE.5860 only; EECE.4860 students may skip to the report)
Reasoning models trade extra generated tokens ("thinking") for accuracy. Qwen3 can switch thinking on and off. On 20 grade-school math problems (GSM8K, Cobbe et al., 2021) you will compare:
1. **direct**: answer immediately with a number;
2. **CoT**: step-by-step reasoning in the answer (thinking off);
3. **thinking**: Qwen3 thinking mode;
4. **self-consistency**: 5 sampled CoT answers, majority vote (Wang et al., 2023).

**Task:** implement `extract_answer` and `majority_vote`, run the experiment, and report accuracy against the number of generated tokens.

In [ ]:
def extract_answer(text):
    """Return the final numeric answer as a normalized string ("1234", "18.5", "-3"), or None.
    Ignore everything inside <think>...</think>; if thinking never finished, return None."""
    # TODO: your code here
    raise NotImplementedError

def majority_vote(answers):
    """Most common non-None answer; ties go to the answer that appeared first. None if all are None."""
    # TODO: your code here
    raise NotImplementedError

if SECTION == "5860":
    ck.expect_fn("C.extract_answer", extract_answer, [
        (("The answer is 72.",), "72"),
        (("<think>2+2=4, so 5?</think>\nSo she has 1,234 apples.",), "1234"),
        (("<think>unfinished reasoning 12",), None),
        (("Total: $18.50",), "18.5"),
        (("No number here",), None),
        (("x = -3",), "-3"),
        (("It costs 10.00 dollars",), "10"),
    ])
    ck.expect_fn("C.majority_vote", majority_vote, [
        ((["5", "3", "5", None],), "5"),
        (([None, None],), None),
        ((["2", "3"],), "2"),
        ((["7", None, "8", "8"],), "8"),
    ])

In [ ]:
if SECTION == "5860":
    from datasets import load_dataset
    gsm = load_dataset("openai/gsm8k", "main", split="test")
    idx = random.Random(V).sample(range(200), 3 if QUICK else 20)
    qs = [gsm[i]["question"] for i in idx]
    gold = [gsm[i]["answer"].split("####")[-1].strip().replace(",", "") for i in idx]
    tok, _ = load(MAIN_MODEL)
    DIRECT = "Answer with only the final number, no explanation.\n\n"
    COT = "Solve step by step, then give the final answer on the last line as 'Answer: <number>'.\n\n"
    budget = 256 if QUICK else 1536
    runs = {}
    with measure("C direct"):
        runs["direct"] = generate(MAIN_MODEL, chat_prompts(tok, [DIRECT + q for q in qs]), max_new_tokens=16, do_sample=False)
    with measure("C CoT"):
        runs["CoT"] = generate(MAIN_MODEL, chat_prompts(tok, [COT + q for q in qs]), max_new_tokens=384, do_sample=False)
    with measure("C thinking"):
        runs["thinking"] = generate(MAIN_MODEL, chat_prompts(tok, [COT + q for q in qs], thinking=True),
                                    max_new_tokens=budget, do_sample=True, temperature=0.6, top_p=0.95, top_k=20)
    with measure("C self-consistency"):
        sc_text, sc_tok = generate(MAIN_MODEL, chat_prompts(tok, [COT + q for q in qs]), max_new_tokens=384,
                                   do_sample=True, temperature=0.7, top_p=0.95, num_return_sequences=5)
    table_c = {}
    for name, (texts, ntok) in runs.items():
        preds = [extract_answer(t) for t in texts]
        table_c[name] = (sum(p == g for p, g in zip(preds, gold)) / len(gold), sum(ntok) / len(ntok))
    votes = [majority_vote([extract_answer(t) for t in sc_text[5 * i:5 * i + 5]]) for i in range(len(qs))]
    table_c["self-consistency(k=5)"] = (sum(v == g for v, g in zip(votes, gold)) / len(gold), sum(sc_tok) / len(qs))
    print(f"{'method':22s}{'accuracy':>10s}{'tokens/question':>18s}")
    for name, (acc, nt) in table_c.items():
        print(f"{name:22s}{acc:10.2f}{nt:18.0f}")
        ck.record(f"C.{name}", {"accuracy": round(acc, 3), "tokens_per_question": round(nt, 1)})
    ck.expect_true("C.thinking_uses_more_tokens", table_c["thinking"][1] > table_c["direct"][1],
                   value=(round(table_c["direct"][1]), round(table_c["thinking"][1])))

## Results summary
Run this cell last. It prints your checks and the results record the grader reads. **Do not edit its output.**

In [ ]:
_ = ck.summary()

## Report (5 points)
Write your answers in the Markdown cells below (double-click to edit). Keep each answer to 2–5 sentences and cite your own numbers from the outputs above.

### 1. Summary
*In your own words, what did this lab show?*

**Your answer:**

### 2. Problems encountered and how you solved them

**Your answer:**

### 3. Questions
**Q1 (A1).** Which tokenizer needed the most tokens for the Spanish and the numbers texts? Why does the token count matter for cost and for the context window?

**Your answer:**

**Q2 (A2–A3).** How did distinct-2 change with temperature and with top-p? Quote one low-temperature and one high-temperature sample, and explain the difference using $p_i \propto e^{z_i/T}$.

**Your answer:**

**Q3 (A3).** Why did greedy decoding produce the same output twice? Name one application where you would prefer greedy decoding and one where you would not.

**Your answer:**

**Q4 (B).** Compare zero-shot, few-shot, and few-shot + retry on validity and field accuracy. Which field was hardest, and why do you think so? Show one wrong extraction.

**Your answer:**

**Q5 (B4).** Was the model's answer about the 2026 World Series correct, a refusal, or a hallucination? What does this tell you about knowledge cutoffs, and how could retrieval-augmented generation (Week 9) help?

**Your answer:**

**Q6 (C, EECE.5860).** Is the extra test-time compute worth it on these problems? Use your accuracy and tokens-per-question numbers.

**Your answer:**

**Q7 (C, EECE.5860).** Find one problem where self-consistency fixed a greedy CoT error (or explain why it never did in your run).

**Your answer:**

### 4. Lessons learned

**Your answer:**

### 5. AI-use disclosure
*Follow the course AI-use policy in the syllabus: name any AI tool you used and how you used it, or write "None".*

**Your answer:**